# Lecture 2: Trajectory Optimization

**Theme:** trajectory optimization is Lecture 1's constrained optimization
toolbox applied to time-indexed variables. The decision variables now include a
sequence of states and controls, and the equality constraints are dynamics.

By the end of this lecture, students should be able to:

- recognize dynamics as equality constraints;
- compare indirect and direct formulations;
- explain why indirect rollout derivatives are dense;
- assemble and solve an LQR problem as one equality-constrained QP;
- interpret time-indexed dynamics multipliers as costates;
- describe nonlinear trajectory optimization as repeated local QP solves.


**Attribution:**
Based on lectures from _Optimal Control and Reinforcement Learning_ as taught in the Robotics Institute at Carnegie Mellon University, licensed under <a href="http://creativecommons.org/licenses/by-nc-sa/4.0/?ref=chooser-v1" target="_blank" rel="license noopener noreferrer" style="display:inline-block;">CC BY-NC-SA 4.0<img style="height:22px!important;margin-left:3px;vertical-align:text-bottom;" src="https://mirrors.creativecommons.org/presskit/icons/cc.svg?ref=chooser-v1"><img style="height:22px!important;margin-left:3px;vertical-align:text-bottom;" src="https://mirrors.creativecommons.org/presskit/icons/by.svg?ref=chooser-v1"><img style="height:22px!important;margin-left:3px;vertical-align:text-bottom;" src="https://mirrors.creativecommons.org/presskit/icons/nc.svg?ref=chooser-v1"><img style="height:22px!important;margin-left:3px;vertical-align:text-bottom;" src="https://mirrors.creativecommons.org/presskit/icons/sa.svg?ref=chooser-v1"></a></p>

```text
Andy Goldschmidt 
Andy.Goldschmidt@jhuapl.edu
Johns Hopkins Applied Physics Laboratory
```

## Setup

Run these cells from the `lecture-2-trajectory-optimization` environment:

```julia
julia --project=.
```



In [1]:
using CairoMakie
using ForwardDiff
using LinearAlgebra
using Optim
#using Printf
#using SparseArrays

# From KKT to Trajectory Optimization

Lecture 1 ended with equality-constrained quadratic programs:

$$
\min_z \frac{1}{2}z^T H z + q^T z
\quad \text{subject to} \quad Cz=d.
$$

The KKT conditions are one linear system:

$$
\begin{bmatrix}
H & C^T \\
C & 0
\end{bmatrix}
\begin{bmatrix}
z \\
\lambda
\end{bmatrix}
=
\begin{bmatrix}
-q \\
d
\end{bmatrix}.
$$

In trajectory optimization, `z` contains variables at many times,
- _Snapshot matrix_, $Z = \begin{bmatrix} | & | & & | \\ z_1 & z_2 & & z_N \\ | & | & & | \end{bmatrix}$
- _Knot point_, $z_1 = \begin{bmatrix} x_1 \\ u_1 \end{bmatrix}$

The dynamics can be posed as equality constraints:

\begin{align*}
\min_{x_{1:N}, u_{1:N{-}1}} &\sum_{n=1}^{N-1} \ell_n(x_n, u_n) + \ell_N(x_N) \\
\text{s.t.} &\quad x_{n+1} = f(x_n, u_n, n)
\end{align*}

We can solve this with a KKT system!

## 1. Two Views of Trajectory Optimization

We will cover two topics:

```text
Indirect optimization:
controls are variables; states come from rollout; dynamics are hidden inside simulation.
optimize-then-discretize viewpoint

Direct optimization:
states and controls are variables; dynamics are sparse constraints. discretize-then-optimize viewpoint
```

These viewpoints often solve related problems, but they expose different structure to the optimizer. Quantum-control methods such as GRAPE are closest to the indirect view.

## 2. Discrete Dynamics and LQR

Start with a dynamical system:

$$
\dot{x} = A_c x + B_c u.
$$

With a zero-order hold and a forward Euler step, a simple discretization is

$$
x_{n+1}=Ax_n+Bu_n.
$$

The double integrator is a simple system following Newton's second law, $F = ma$.


$$
m \frac{d}{dt} \begin{bmatrix} q \\ \dot{q} \end{bmatrix} 
= \begin{bmatrix} 0 & m \\ 0 & 0 \end{bmatrix} \begin{bmatrix} q \\ \dot{q} \end{bmatrix}
+ \begin{bmatrix} 0 \\ u \end{bmatrix}
$$

Let $x = \begin{bmatrix}q \\ \dot{q}\end{bmatrix}.$


In [2]:
dt = 0.1
N = 50

function double_integrator(m)
    Ac = [0.0 1.0; 0.0 0.0]
    Bc = [0.0; 1.0 / m]
    return Ac, Bc
end

Ac, Bc = double_integrator(1.0)

A = I(2) + dt * Ac
B = dt * Bc

A, B

([1.0 0.1; 0.0 1.0], [0.0, 0.1])

- Matrix exponential trick for linear systems

$$
\exp\left(\begin{bmatrix} A & B \\ 0 & 0 \end{bmatrix} h \right)
= \begin{bmatrix} A_h & B_h \\ 0 & I \end{bmatrix}
$$

- Compare
\begin{align*}
\dot{x}(t) &= A x(t) + B u(t)  \\
{\Rightarrow}\quad x(t+h) &=  \left( \sum_{n\ge0} \tfrac{1}{n!} A^n h^n \right) x + \left( \sum_{n\ge1} \tfrac{1}{n!} A^{n-1} B h^n \right) u \\
&\approx (I + h A) x + h B u
\end{align*}

In [3]:
function continuous_to_discrete(Ac, Bc, h)
    # Construct augmented matrix for matrix exponential
    augmented_matrix = [
        Ac Bc; 
        zeros(size(Bc, 2), size(Ac, 1)) zeros(size(Bc, 2), size(Bc, 2))
    ]

    # Compute matrix exponential
    exp_matrix = exp(augmented_matrix * h)

    # Extract discrete LTI system matrices
    Ah = exp_matrix[1:size(Ac, 1), 1:size(Ac, 2)]
    Bh = exp_matrix[1:size(Ac, 1), size(Ac, 2)+1:end]

    return Ah, Bh
end

continuous_to_discrete (generic function with 1 method)

In [4]:
# Matrix trick
A, B = continuous_to_discrete(Ac, Bc, dt)
A, B

([1.0 0.1; 0.0 1.0], [0.005000000000000001; 0.1;;])

In [5]:
# Euler
A = I(2) + dt * Ac
B = dt * Bc
A, B

([1.0 0.1; 0.0 1.0], [0.0, 0.1])

In [6]:
function simulate_dlti(u, A, B, x1)
    N = length(u) + 1
    x = zeros(eltype(u), length(x1), N)
    x[:, 1] .= x1

    for n in 1:(N - 1)
        x[:, n + 1] .= A * x[:, n] + B * u[:, n]
    end

    return x
end

simulate_dlti (generic function with 1 method)

### Linear Quadratic Regulator

The LQR problem is

$$
\min_{x_{1:N},u_{1:N-1}}
\sum_{n=1}^{N-1}
\frac{1}{2}x_n^TQx_n + \frac{1}{2}u_n^TRu_n
+ \frac{1}{2}x_N^TQ_Nx_N
$$

subject to

$$
x_{n+1}=Ax_n+Bu_n.
$$

This is a convex equality-constrained QP when `Q` and `Qf` are positive
semidefinite and `R` is positive definite.



In [ ]:
function lqr_cost(x, u, x1, xgoal, Q, R, Qf)
    J = zero(eltype(u))

    for n in 1:(size(x, 2) - 1)
        J += 0.5 * dot(x[:, n] - xgoal, Q * (x[:, n] - xgoal))
        J += 0.5 * dot(u[:, n], R * u[:, n])
    end

    J += 0.5 * dot(x[:, end] - xgoal, Qf * (x[:, end] - xgoal))
    return J
end

## 3. Indirect trajectory optimization

```text
Let's just get rid of that equality constraint.
```

### Single Shooting / GRAPE

In indirect form, the controls are the only decision variables:

$$
\min_u J(x(u),u).
$$

The states are generated by rollout, so the dynamics are always satisfied.

This is conceptually close to GRAPE in quantum control: the optimizer changes a
control waveform, a simulator produces the resulting evolution, and gradients
propagate through that time-ordered rollout.



In [ ]:
function indirect_lqr_cost(u, A, B, x1, xgoal, Q, R, Qf)
    x = simulate_dlti(u, A, B, x1)
    J = zero(eltype(u))

    return lqr_cost(x, u, x1, xgoal, Q, R, Qf)
end

In [ ]:
dt = 0.1
N = 50

A, B = continuous_to_discrete(Ac, Bc, dt)

x1 = [-1.0, -0.2]
xgoal_lqr = [0.0, 0.0]

Q = Diagonal([1.0, 1e-1])
R = 0.1
Qf = Diagonal([10.0, 1.0])

In [ ]:
u0 = zeros(1, N - 1)

indirect_J(u) = indirect_lqr_cost(u, A, B, x1, xgoal_lqr, 100 *Q, R, Qf)
indirect_∇J!(g, u) = (g .= ForwardDiff.gradient(indirect_J, u))

result = optimize(
    indirect_J, indirect_∇J!, u0, LBFGS(), Optim.Options(iterations=200)
)
u_indirect = Optim.minimizer(result)
x_indirect = simulate_dlti(u_indirect, A, B, x1)

Optim.minimum(result), x_indirect[:, end]

In [ ]:
function plot_phase_trajectory(x; limits=(nothing, nothing), title="trajectory")
    fig = Figure(size=(640, 440))
    ax = Axis(fig[1, 1], xlabel="position", ylabel="velocity", limits=limits, title=title, aspect=1)
    lines!(ax, x[1, :], x[2, :], linewidth=2)
    scatter!(ax, [x[1, 1]], [x[2, 1]], color=:green, markersize=16, label="start")
    scatter!(ax, [x[1, end]], [x[2, end]], color=:white, strokecolor=:black, strokewidth=2, markersize=14, label="finish")
    axislegend(ax; position=:rt)
    return fig
end

function plot_control(u; title="control")
    fig = Figure(size=(640, 440))
    ax = Axis(fig[1, 1], xlabel="timestep", ylabel="control", title=title)
    stairs!(ax, u[1, :], linewidth=2)
    return fig
end

In [ ]:
plot_phase_trajectory(x_indirect; title="Single-shooting LQR trajectory")

In [ ]:
plot_control(u_indirect, title="Single-shooting Force")

**Pause for discussion**

- What variables did the optimizer see?
- Why are the dynamics exactly satisfied?
- If an early control changes, how many future states change?



In [ ]:
u2 = copy(u_indirect)
u2[end] *= 0.95

plot_phase_trajectory(
    simulate_dlti(u2, A, B, x1); title="Single-shooting LQR trajectory"
)

### Dense Reduced Derivatives

For linear dynamics,

$$
x_{n+1}=Ax_n+Bu_n,
$$

a control affects every later state:

$$
\frac{\partial x_n}{\partial u_k}=A^{n-1-k}B,\qquad k<n.
$$

The reduced problem has fewer variables, but rollout derivatives become dense
across time.



In [ ]:
function rollout_vector(u)
    return vec(simulate_dlti(u, A, B, x1))
end

∂F = ForwardDiff.jacobian(rollout_vector, u0)

fig = Figure(size=(720, 420))
ax = Axis(fig[1, 1], yreversed=true, aspect=1, 
    xlabel="control index", ylabel="state-vector row", title="Jacobian sparsity"
)
heatmap!(ax, abs.(∂F) .> 1e-10; colormap=[:white, :black])
fig

We'll revisit this when we talk about Pontryagin!

## 4. Direct trajectory optimization

In direct form, we keep states and controls as decision variables:

\begin{align*}
\min_{x_{1:N}, u_{1:{N{-}1}}} &\quad J(x_{1:N}, u_{1:{N{-}1}}) = \sum_{n=1}^{N-1} \tfrac{1}{2} x_n^T Q_n x_n + \tfrac{1}{2} u_n^T R_n u_n + \tfrac{1}{2} x_N^T Q_N x_N \\
\text{s.t.} &\quad x_{n+1} = A_n x_n + B_n u_n 
\end{align*}
<!-- $$
\min_{x,u} J(x,u)
\quad \text{subject to} \quad
x_{n+1}=Ax_n+Bu_n.
$$ -->

Package all variables into one vector `z`, then build

$$
\boxed{
\min_z \frac{1}{2}z^THz+q^Tz
\quad \text{subject to} \quad Cz=d.
}
$$

This is **direct transcription**: first discretize the dynamics, then optimize
over the discrete trajectory. The dynamics residuals are often called
**defect constraints**. Direct collocation uses richer integration/collocation
rules, but the optimization structure is the same: trajectory variables plus
sparse local constraints.



\begin{align*}
Z &= \begin{bmatrix} 
    x_1 & x_2 & \cdots & x_N \\
    u_1 & u_2 & \cdots & u_N 
\end{bmatrix} \\
\Rightarrow \vec{Z} &= \begin{bmatrix} x_1 \\ u_1 \\ x_2 \\ u_2 \\ \vdots \\ x_N \\ u_N \end{bmatrix}
\end{align*}

and drop the first state (known) and last control (not used),
$$
 z = \vec{Z}[\texttt{length}(x_1){:}\texttt{end}{-}\texttt{length}(u_N)]
$$

In [ ]:
function trajectory_indices(nx, nu, N)
    xoffset(n) = (n - 1) * nx
    uoffset(n) = nx * N + (n - 1) * nu
    xidx(n) = (xoffset(n) + 1):(xoffset(n) + nx)
    uidx(n) = (uoffset(n) + 1):(uoffset(n) + nu)
    return xidx, uidx
end

$\tfrac{1}{2} z^T H z$
\begin{equation}
    H = \begin{bmatrix}
        R_1 & 0 & 0 & & 0 \\
        0 & Q_2 & 0 & \cdots & 0 \\
        0 & 0 & R_2 & & 0 \\
        & \vdots & & \ddots & \vdots \\
        0 & 0 & 0 & \cdots & Q_N \\
    \end{bmatrix}
\end{equation}

In [ ]:
function assemble_qp_objective(Q, R, Qf, xgoal, N)
    nx = size(Q, 1)
    nu = size(R, 1)
    nz = nx * N + nu * (N - 1)

    xidx, uidx = trajectory_indices(nx, nu, N)

    H = spzeros(nz, nz)
    q = zeros(nz)

    for n in 1:(N - 1)
        H[xidx(n), xidx(n)] .= Q
        H[uidx(n), uidx(n)] .= R * I(nu)
        q[xidx(n)] .= -Q * xgoal
    end

    H[xidx(N), xidx(N)] .= Qf
    q[xidx(N)] .= -Qf * xgoal

    return H, q
end



$Cz = d$
\begin{equation}
C = \begin{bmatrix}
    B_1 & -I & 0 & 0 & & 0 \\
    0 & A_2 & B_2 & -I & \cdots & 0 \\
    \vdots & \vdots & \ddots & \ddots & \ddots & 0 \\
    0 & 0 & \cdots & A_{N-1} & B_{N-1} & -I
\end{bmatrix}, \qquad
d = \begin{bmatrix}
    -A_1 x_1 \\
    0 \\
    0 \\
    \vdots \\
    0
\end{bmatrix}
\end{equation}


In [ ]:
function assemble_trajectory_constraints(As, Bs, x1, defects)
    N = length(As) + 1
    nx = size(As[1], 1)
    nu = size(Bs[1], 2)

    nz = nx * N + nu * (N - 1)
    nc = nx * N
    
    C = spzeros(nc, nz)
    d = zeros(nc)

    xidx, uidx = trajectory_indices(nx, nu, N)

    # Initial state or initial-state increment
    C[1:nx, xidx(1)] .= I(nx)
    d[1:nx] .= x1

    for n in 1:(N - 1)
        rows = (nx * n + 1):(nx * (n + 1))

        C[rows, xidx(n)] .= -As[n]
        C[rows, uidx(n)] .= -Bs[n]
        C[rows, xidx(n + 1)] .= I(nx)

        d[rows] .= defects[n]
    end

    return C, d
end

Create and solve the QP. We need the Lagrangian and its derivatives.

Lagrangian:
\begin{equation*}
    L(z, \lambda) = \tfrac{1}{2} z^T H z + \lambda^T (C z - d)
\end{equation*}

KKT conditions:
\begin{align*}
    & \nabla_z L = H z + C^T \lambda \overset{!}{=} 0 \\
    & \nabla_\lambda L = Cz - d \overset{!}{=} 0 \\
\end{align*}

\begin{equation*}
    \Rightarrow \begin{bmatrix} H & C^T \\ C & 0 \end{bmatrix} 
    \begin{bmatrix} z \\ \lambda \end{bmatrix} 
    = \begin{bmatrix} 0 \\ d \end{bmatrix}
\end{equation*}

In [ ]:
function assemble_lqr_qp(A, B, x1, xgoal, Q, R, Qf, N)
    nx = length(x1)
    nu = size(B, 2)
    nz = nx * N + nu * (N - 1)

    As = [A for _ in 1:(N - 1)]
    Bs = [B for _ in 1:(N - 1)]
    # The dynamics are feasible, but they don't have to be
    defects = [zeros(nx) for _ in 1:(N - 1)]

    C, d = assemble_trajectory_constraints(As, Bs, x1, defects)
    
    H, q = assemble_qp_objective(Q, R, Qf, xgoal, N)

    return (
        H=H, 
        q=q, 
        C=C, 
        d=d
    )
end

In [ ]:
qp = assemble_lqr_qp(A, B, x1, xgoal_lqr, Q, R, Qf, N);

How many iterations will this take to solve?

In [ ]:
function solve_equality_qp(H, q, C, d)
    K = [H C'; C spzeros(size(C, 1), size(C, 1))]
    rhs = [-q; d]
    sol = K \ rhs
    z = sol[1:size(H, 1)]
    lambda = sol[(size(H, 1) + 1):end]
    return z, lambda
end

In [ ]:
z_direct, lambda_direct = solve_equality_qp(qp.H, qp.q, qp.C, qp.d)
xidx, uidx = trajectory_indices(length(x1), size(B, 2), N)

x_direct = stack([z_direct[xidx(n)] for n in 1:N])
u_direct = stack([z_direct[uidx(n)] for n in 1:(N - 1)])

# Compare to previous
norm(x_direct - x_indirect), norm(u_direct - u_indirect)

What about the temporal structure?

In [ ]:
fig = Figure(size=(720, 420))
ax = Axis(fig[1, 1], 
    aspect=1, yreversed=true,
    xlabel="variable index", ylabel="constraint row", title="Direct dynamics sparsity")
heatmap!(ax, abs.(qp.C) .> 1e-10; colormap=[:white, :black])
fig

The direct problem uses more variables, but gives the solver sparse local
structure. Each dynamics row touches only `x_n`, `u_n`, and `x_{n+1}`. This is
why direct methods can naturally add local constraints such as amplitude
bounds, smoothness penalties, state constraints, leakage penalties, or terminal
constraints.



In [ ]:
plot_phase_trajectory(x_direct; title="Direct QP trajectory")

In [ ]:
plot_control(u_direct, title="Direct QP Force")

## 6. Indirect (again)

```text
In which we use the temporal structure to avoid deep gradients.
```


### Discretizing Pontryagin's Principle

Instead of eliminating the dynamics contraint, follow the KKT formula. For regularizing towards a goal ($\Delta x_n = x_n - x_g$),  the Lagrangian is:

$$
L =
\sum_{n=1}^{N-1}
\left(
\frac{1}{2}(\Delta x_n)^TQ( \Delta x_n )
+\frac{1}{2}u_n^TRu_n
+\lambda_{n+1}^T(x_{n+1}-Ax_n-Bu_n)
\right)
+\frac{1}{2}(\Delta x_N)^TQ_N \Delta x_N.
$$


KKT conditions:

\begin{align*}
    \frac{\partial L}{\partial \lambda_n} &= (A x_n + B u_n - x_{n+1})^T \overset{!}{=} 0 \\
    \frac{\partial L}{\partial x_n} &= x_n^T Q + \lambda_{n+1}^T A - \lambda_{n}^T \overset{!}{=} 0 \\
    \frac{\partial L}{\partial x_N} &= x_N^T Q_N - \lambda_{N}^T \overset{!}{=} 0 \\
    \frac{\partial L}{\partial u_n} &= u_n^T R + \lambda_{n+1}^T B \overset{!}{=} 0
\end{align*}

We can rewrite the KKT conditions so they look like dynamics:

\begin{align}
    x_{n+1} &= A x_n + B u_n \\
    \lambda_{n} &= A^T \lambda_{n+1} + Q x_n \\
    \lambda_N &= Q_N x_N \\
    u_n &= -R^{-1} B^T \lambda_{n+1}
\end{align}

**(1) Backward solve** We can integrate $\lambda_{n} = A^T \lambda_{n+1} + Q x_n$ backward from $\lambda_N = Q_N x_N$ using $x_{1:N}$.

**(2) Forward solve** Given $x_1$, we can integrate $x_{n+1} = A x_n + B u_n$ forward in time using the policy $u_n = -R^{-1} B^T \lambda_{n+1}$ and the Lagrange multipliers, $\lambda_{1:N}$. 

**Pause for discussion**
- How does this computation differ from what we did prevously? (How are we allocating memory and compute resources?)

In [ ]:
function simulate_dlti_adjoint(x, A, Q, Qf, xgoal)
    N = size(x, 2)
    λ = zeros(size(x))

    λ[:, N] .= Qf * (x[:, N] - xgoal)
    
    for n in (N - 1):-1:1
        λ[:, n] .= A' * λ[:, n + 1] + Q * (x[:, n] - xgoal)
    end

    return λ
end

In [ ]:
function backtracking_control_step(
    objective,
    u,
    du,
    directional_derivative;
    armijo_constant = 1e-4,
    backtracking_factor = 0.5,
    minimum_step_size = 1e-6,
)
    α = 1.0
    current_cost = objective(u)

    while α >= minimum_step_size
        candidate_u = u .+ α .* du
        candidate_cost = objective(candidate_u)

        armijo_bound =
            current_cost +
            armijo_constant * α * directional_derivative

        candidate_cost <= armijo_bound && return α

        α *= backtracking_factor
    end

    return 0.0
end

In [ ]:
@kwdef struct PontryaginSolverParams
    control_tolerance = 1e-2
    max_iters = 500
    verbose = false
end

In [ ]:
function solve_lqr_pontryagin(
    A,
    B,
    x1,
    xgoal,
    N;
    Q = 1e-4I,
    R = 1e-1,
    Qf = 1e2I,
    params = PontryaginSolverParams(),
)
    # Initial trajectory
    u = zeros(1, N - 1)
    x = simulate_dlti(u, A, B, x1)

    λ = zeros(size(x))
    ∇Jᵤ = zeros(size(u))
    Δu = fill(Inf, size(u))

    lqr_params = (x1, xgoal, Q, R, Qf)

    # Define the reduced objective as a rollout 
    reduced_lqr_cost(candidate_u) = begin
        candidate_x = simulate_dlti(candidate_u, A, B, x1)
        lqr_cost(candidate_x, candidate_u, lqr_params...)
    end

    iteration = 0

    while norm(Δu, Inf) > params.control_tolerance && iteration < params.max_iters

        params.verbose && println("Iteration: ", iteration)

        # Backward pass
        λ = simulate_dlti_adjoint(x, A, Q, Qf, xgoal)

        # Obtain the step and the directional derivative
        for n in 1:(N - 1)
            # Gradient of the cost with respect to uₙ
            ∇Jᵤ[:, n] .= R .* u[:, n] + B' * λ[:, n + 1]

            # Preconditioned negative-gradient direction
            Δu[:, n] .= -∇Jᵤ[:, n] ./ R
        end
        directional_derivative = dot(∇Jᵤ, Δu)

        # Stop before performing another rollout if the update is small
        update_norm = norm(Δu, Inf)
        update_norm <= params.control_tolerance && break

        # Armijo line search
        α = backtracking_control_step(reduced_lqr_cost, u, Δu, directional_derivative)

        if α == 0
            params.verbose && println("Line search failed.")
            break
        end

        u .= u .+ α .* Δu
        x = simulate_dlti(u, A, B, x1)

        params.verbose && println(
            "  cost = ", lqr_cost(x, u, lqr_params...),
            ", ‖Δu‖∞ = ", update_norm,
            ", α = ", α,
        )

        iteration += 1
    end

    # Ensure the returned costates correspond to the returned trajectory.
    λ = simulate_dlti_adjoint(x, A, Q, Qf, xgoal)

    return (
        x = x,
        u = u,
        costates = λ,
        cost = lqr_cost(x, u, lqr_params...),
        iterations = iteration,
    )
end

In [ ]:
sol = solve_lqr_pontryagin(A, B, x1, zeros(size(x1)), N, Q=100 * Q, R=1e0 * R, Qf=Qf)

x_pontryagin = sol.x
u_pontryagin = sol.u

# Compare to previous
norm(x_pontryagin - x_indirect), norm(u_pontryagin - u_indirect)

In [ ]:
plot_phase_trajectory(x_pontryagin; title="Indirect Pontryagin Trajectory")

In [ ]:
plot_control(u_pontryagin, title="Indirect Pontryagin Force")

## 7. Nonlinear Trajectory Optimization

LQR was a one-step QP because the dynamics were linear and the cost was quadratic. For
**nonlinear dynamics**, we can still build a local linear-quadratic approximation, solve a 
local QP, line search, and repeat.

Quantum dynamics have a bilinear control structure, for example

$$
\partial_t \ket{\psi}= -i(H_0+u(t)H_1) \ket{\psi},
$$

or, for density matrices,

$$
\dot{\rho}=-i[H_0+u(t)H_1,\rho]+\text{dissipators}.
$$


### Rotations
The real-valued example below keeps the same bilinear shape without introducing quantum 
notation or package abstractions.

Consider a damped bilinear rotation:

$$
\dot{x}=uAx-\gamma x,
\qquad
A = \begin{bmatrix}0 & -1 \\ 1 & 0\end{bmatrix}.
$$

With forward Euler,

$$
x_{n+1}=x_n+h(u_nAx_n-\gamma x_n).
$$

For `gamma = 0`, the continuous dynamics are a pure rotation and preserve
`norm(x)`, but discretization can change that! Forward Euler does not preserve that 
geometry: nonzero controls grow the norm. Backward Euler actually has the opposite bias and 
shrinks the norm. This is a useful warning for quantum control, where the exact Schrödinger 
evolution is unitary and norm-preserving.

#### Linearization

When we linearize around a nominal trajectory, we lose our model's time invariance:

$$
\Delta x_{n+1}\approx A_n\Delta x_n+B_n\Delta u_n,
$$

where

$$
A_n=I-h\gamma I+h u_n A,
\qquad
B_n=hAx_n.
$$



In [ ]:
γ = 0.00
Anl = [0.0 -1.0; 1.0 0.0]
r_nl = 1e-3

function bilinear_rotation_step(xn, un; h=dt, γ=γ, A=Anl)
    return xn .+ h .* (un .* (A * xn) .- γ .* xn)
end

function simulate_nonlinear(u, x1)
    N = size(u, 2) + 1
    x = zeros(promote_type(eltype(u), eltype(x1)), length(x1), N)
    x[:, 1] .= x1

    for n in 1:(N - 1)
        x[:, n + 1] .= bilinear_rotation_step(x[:, n], u[1, n])
    end

    return x
end

function nonlinear_objective(u, x1, xgoal; r=r_nl)
    x = simulate_nonlinear(u, x1)
    terminal_error = x[:, end] - xgoal
    return 0.5 * dot(terminal_error, terminal_error) + 0.5 * r * dot(u, u)
end

In [ ]:
function local_bilinear_models(x, u; h=dt, γ=γ, A=Anl)
    N = size(x, 2)
    nx = size(x, 1)

    As = Matrix{eltype(x)}[]
    Bs = Matrix{promote_type(eltype(x), eltype(u))}[]

    for n in 1:(N - 1)
        push!(As, Matrix(I, nx, nx) .+ h .* (u[1, n] .* A .- γ .* I(nx)))
        push!(Bs, reshape(h .* A * x[:, n], nx, 1))
    end

    return As, Bs
end

function trajectory_defects(x, u)
    N = size(x, 2)
    return [
        bilinear_rotation_step(x[:, n], u[1, n]) - x[:, n + 1]
        for n in 1:(N - 1)
    ]
end

In [ ]:
function assemble_increment_objective(x, u, xgoal; r=r_nl, μ=1e-3)
    nx = size(x, 1)
    nu = size(u, 1)
    N = size(x, 2)
    nz = nx * N + nu * (N - 1)

    xidx, uidx = trajectory_indices(nx, nu, N)

    H = spzeros(nz, nz)
    q = zeros(nz)

    H[xidx(N), xidx(N)] .= I(nx)
    q[xidx(N)] .= x[:, end] - xgoal

    for n in 1:(N - 1)
        H[uidx(n), uidx(n)] .= (r + μ) .* I(nu)
        q[uidx(n)] .= r .* u[:, n]
    end

    return H, q
end

function assemble_increment_qp(x, u, xgoal; r=r_nl, μ=1e-3)
    nx = size(x, 1)
    As, Bs = local_bilinear_models(x, u)
    defects = trajectory_defects(x, u)

    C, d = assemble_trajectory_constraints(As, Bs, zeros(nx), defects)
    H, q = assemble_increment_objective(x, u, xgoal; r=r, μ=μ)

    return (H=H, q=q, C=C, d=d)
end

In [ ]:
function sqp_step(x, u, xgoal; r=r_nl, μ=1e-3)
    qp = assemble_increment_qp(x, u, xgoal; r=r, μ=μ)
    Δz, λ = solve_equality_qp(qp.H, qp.q, qp.C, qp.d)

    nx = size(x, 1)
    nu = size(u, 1)
    N = size(x, 2)

    xidx, uidx = trajectory_indices(nx, nu, N)

    Δx = stack([Δz[xidx(n)] for n in 1:N])
    Δu = stack([Δz[uidx(n)] for n in 1:(N - 1)])
    DJ = dot(qp.q, Δz)

    return (
        Δx=Δx,
        Δu=Δu,
        Δz=Δz,
        DJ=DJ,
        multipliers=λ,
    )
end

@kwdef struct SQPParams
    control_tolerance = 1e-3
    maximum_iterations = 50
    regularization = 1e-3
    armijo_constant = 1e-4
    backtracking_factor = 0.5
    minimum_step_size = 1e-6
    verbose = false
end

function solve_sqp(u0, x1, xgoal; r=r_nl, params=SQPParams())
    u = copy(u0)
    x = simulate_nonlinear(u, x1)
    cost = nonlinear_objective(u, x1, xgoal; r=r)
    iteration = 0

    for k in 1:params.maximum_iterations
        step = sqp_step(x, u, xgoal; r=r, μ=params.regularization)
        update_norm = norm(step.Δu, Inf)
        update_norm <= params.control_tolerance && break

        objective(candidate_u) = nonlinear_objective(candidate_u, x1, xgoal; r=r)
        α = backtracking_control_step(objective, u, step.Δu, step.DJ)

        if α == 0
            params.verbose && println("Line search failed.")
            break
        end

        u .= u .+ α .* step.Δu
        x = simulate_nonlinear(u, x1)
        cost = objective(u)
        iteration = k

        params.verbose && println(
            "Iteration: ", k,
            ", cost = ", cost,
            ", ‖Δu‖∞ = ", update_norm,
            ", α = ", α,
        )
    end

    return (x=x, u=u, cost=cost, iterations=iteration)
end

In [ ]:
ϕ = -π/2.5
# ϕ = -π/2
x1_nl = [cos(ϕ), sin(ϕ)]
xgoal_nl = [0.0, 1.0]
u0_nl = zeros(1, N - 1)

sqp_sol = solve_sqp(
    u0_nl,
    x1_nl,
    xgoal_nl;
    params=SQPParams(verbose=true, maximum_iterations=100),
)

sqp_sol.cost, sqp_sol.x[:, end], sqp_sol.iterations

In [ ]:
plot_phase_trajectory(sqp_sol.x; limits=((-1.2, 1.2), (-1.2, 1.2)), title="SQP nonlinear trajectory")

In [ ]:
plot_control(sqp_sol.u; title="SQP nonlinear control")

**Pause for discussion**
- Why is the control unable to exactly reach the goal?
- What happens at $\pi/2$?